# Elevación e indicadores de las rutas piloto

## Objetivo

Este notebook verifica los perfiles obtenidos con Copernicus DEM GLO-30 Public, compara escenarios de filtrado y reúne los indicadores preliminares de las cuatro rutas piloto. Es una etapa histórica, no el inventario completo: las 22 rutas y su elegibilidad vigente se resumen en `04_ampliacion_rutas.ipynb`.

In [1]:
from pathlib import Path
import pandas as pd

actual = Path.cwd().resolve()
if (actual / "01_documentacion").exists():
    proyecto = actual
elif (actual.parent / "01_documentacion").exists():
    proyecto = actual.parent
else:
    raise FileNotFoundError("No se encontró la carpeta principal del proyecto")

carpeta_tablas = proyecto / "07_resultados" / "tablas"
carpeta_perfiles = proyecto / "07_resultados" / "perfiles"
print("Proyecto:", proyecto)

Proyecto: C:\Users\LENOVO\Documents\ChatGPT\tesis_trekking_pichincha


## Fuente y preparación

Se utilizó el mosaico Copernicus DEM GLO-30 Public `S01_00_W079_00`, que cubre completamente las cuatro rutas. Los trazados se proyectaron temporalmente a EPSG:32717 y se dividieron en intervalos uniformes de aproximadamente 30 m.

Durante esta etapa se detectó que R001 terminaba antes de la cumbre. La vía OSM 377339146 se añadió mediante un procedimiento documentado, con lo que la distancia pasó de 4.436 km a 4.804 km por trayecto.

In [2]:
indicadores = pd.read_csv(
    carpeta_tablas / "indicadores_elevacion_piloto.csv"
)

columnas_principales = [
    "id_ruta",
    "distancia_ida_vuelta_km",
    "puntos_perfil",
    "valores_faltantes",
    "elevacion_min_m",
    "elevacion_max_m",
    "ascenso_ida_vuelta_m",
    "pendiente_media_abs_pct",
    "pendiente_p95_abs_pct",
]

assert len(indicadores) == 4
assert indicadores["valores_faltantes"].eq(0).all()
indicadores[columnas_principales]

,id_ruta,distancia_ida_vuelta_km,puntos_perfil,valores_faltantes,elevacion_min_m,elevacion_max_m,ascenso_ida_vuelta_m,pendiente_media_abs_pct,pendiente_p95_abs_pct
0,R001,9.607,162,0,3954.548,4678.573,824.839,17.171,56.440
1,R003,7.484,126,0,3062.865,3289.859,448.981,11.998,43.670
2,R005,1.903,33,0,1410.106,1505.709,165.417,17.388,58.109
3,R006,5.232,89,0,2517.140,3181.260,817.832,31.262,83.923


## Filtrado y sensibilidad

Los valores raw conservan oscilaciones celda a celda que incrementan artificialmente el ascenso acumulado. El método principal provisional aplica una mediana centrada de tres puntos y conserva sin cambios los extremos. La mediana de cinco puntos se mantiene como escenario de mayor suavizado para evaluar sensibilidad.

In [3]:
sensibilidad = pd.read_csv(
    carpeta_tablas / "sensibilidad_elevacion_piloto.csv"
)
comparacion = sensibilidad.pivot(
    index="id_ruta",
    columns="metodo",
    values="ascenso_ida_vuelta_m",
)
assert comparacion.notna().all().all()
comparacion

metodo,mediana_3,mediana_5,raw
id_ruta,,,
R001,824.839,755.419,1061.101
R003,448.981,402.735,654.703
R005,165.417,111.551,336.397
R006,817.832,798.648,863.548


## Revisión de pendientes extremas

Las pendientes máximas puntuales superan el 100 % en R003 y R006. Estos valores corresponden a cambios de una celda en sectores escarpados y se conservan como señales de revisión, no como errores eliminados automáticamente. Para caracterizar la exigencia de la ruta se reportan también la pendiente media absoluta y el percentil 95, menos sensibles a un único tramo extremo.

In [4]:
tramos_extremos = pd.read_csv(
    carpeta_tablas / "tramos_pendiente_extrema_piloto.csv"
)
archivos_perfil = sorted(carpeta_perfiles.glob("R*_perfil_elevacion_2026-10-05.svg"))

assert len(tramos_extremos) == 20
assert len(archivos_perfil) == 4
print("Perfiles gráficos:", [archivo.name for archivo in archivos_perfil])
tramos_extremos.groupby("id_ruta").first()[
    ["distancia_fin_m", "cambio_elevacion_m", "pendiente_pct"]
]

Perfiles gráficos: ['R001_perfil_elevacion_2026-10-05.svg', 'R003_perfil_elevacion_2026-10-05.svg', 'R005_perfil_elevacion_2026-10-05.svg', 'R006_perfil_elevacion_2026-10-05.svg']


,distancia_fin_m,cambio_elevacion_m,pendiente_pct
id_ruta,,,
R001,4684.316,27.201,91.168
R003,3742.078,31.301,104.560
R005,118.914,-22.459,-75.548
R006,386.465,30.515,102.647


## Conclusión provisional

Los cuatro perfiles del piloto tienen cobertura completa y no presentan elevaciones faltantes. Esta es una conclusión histórica sobre el procesamiento de elevación, no una declaración de elegibilidad final: en el inventario actual R001 se excluye del modelado de trekking por corresponder a alta montaña; R003 y R006 son elegibles; R005 sigue pendiente por la identidad y alcance de su variante.

La mediana de tres puntos se conserva como método principal provisional. La decisión final deberá considerar la comparación con la mediana de cinco puntos y la influencia de los tramos extremos.